# Module 5 · Multi-Modal AI: Vision, Audio & Beyond

**Track:** Main Track — GenAI / LLM  
**Toolchain:** CLIP · Whisper · SentenceTransformers · Pillow · GPT-4V API  
**Prerequisites:** NB22 (RAG Pipeline), NB21 (Vector Databases)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 120–150 minutes

---

## Why Multi-Modal Matters in 2026

Text-only AI is leaving 80% of the world's data on the table. Documents contain images, charts, tables. Customer interactions include voice and video. Production AI systems must *see* and *hear*, not just read.

| Application | Modalities | Business Value |
|------------|-----------|---------------|
| Document processing | Image + Text | Extract data from invoices, receipts, forms |
| Medical imaging | Image + Text | Analyze X-rays, describe findings |
| Customer support | Audio + Text | Transcribe calls, analyze sentiment |
| Visual search | Image + Text | "Find products that look like this" |
| Video analysis | Video + Audio + Text | Meeting summaries, content moderation |

### Multi-Modal Models (2026 Landscape)

| Model | Text | Image | Audio | Video | Open Source? |
|-------|:----:|:-----:|:-----:|:-----:|:----:|
| **GPT-4o** | ✅ | ✅ | ✅ | ❌ | ❌ |
| **Gemini 2.0** | ✅ | ✅ | ✅ | ✅ | ❌ |
| **Claude 3.5** | ✅ | ✅ | ❌ | ❌ | ❌ |
| **Llama 3.2 Vision** | ✅ | ✅ | ❌ | ❌ | ✅ |
| **CLIP (ViT-L/14)** | ✅ | ✅ | ❌ | ❌ | ✅ |
| **Whisper v3** | ❌ | ❌ | ✅ | ❌ | ✅ |

**This notebook focuses on the open-source stack** — CLIP and Whisper — because these are the tools you deploy yourself. API-based vision (GPT-4V, Gemini) is covered in the production patterns section.

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors think multi-modal means "send an image to GPT-4V." Seniors know that production multi-modal AI runs on *embeddings* — CLIP maps images and text into the same 768-dimensional space, enabling zero-shot classification, cross-modal search, and multi-modal RAG without any API calls.

**Mental Model:** Imagine a universal translator that converts any language (English, images, audio) into a single "meaning language" — a 768-dimensional vector. Two things that *mean* the same thing (a photo of a dog, the word "dog") become the same vector. This is what CLIP does for images and text.

**Common Junior Pitfall:** Sending full 4K resolution images to a Vision API for simple classification tasks. A $0.15 API call per image adds up to $15K for 100K images. Seniors use CLIP for classification (free, local, 50ms) and reserve Vision APIs for complex reasoning.

---

## 📑 Table of Contents

* [Why Multi-Modal Matters in 2026](#why-multi-modal-matters-in-2026)
  * [Multi-Modal Models (2026 Landscape)](#multi-modal-models-2026-landscape)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. CLIP: Image-Text Embeddings](#1-clip-image-text-embeddings)
  * [1.1 How CLIP Works (Contrastive Learning)](#11-how-clip-works-contrastive-learning)
  * [1.2 Hands-On: Loading CLIP and Computing Embeddings](#12-hands-on-loading-clip-and-computing-embeddings)
  * [1.3 Zero-Shot Image Classification](#13-zero-shot-image-classification)
  * [1.4 Cross-Modal Search (Image ↔ Text)](#14-cross-modal-search-image-text)
* [2. Whisper: Production Speech-to-Text](#2-whisper-production-speech-to-text)
  * [2.1 Whisper Architecture Overview](#21-whisper-architecture-overview)
  * [2.2 Hands-On: Transcription with Whisper](#22-hands-on-transcription-with-whisper)
  * [2.3 Production Audio Pipeline](#23-production-audio-pipeline)
* [3. Vision API Patterns (GPT-4V / Gemini)](#3-vision-api-patterns-gpt-4v-gemini)
  * [3.1 Image Input Formats and Cost Optimization](#31-image-input-formats-and-cost-optimization)
  * [3.2 Structured Extraction from Images](#32-structured-extraction-from-images)
* [4. Multi-Modal RAG](#4-multi-modal-rag)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Why is CLIP better than GPT-4V for image classification at scale?](#q1-why-is-clip-better-than-gpt-4v-for-image-classification-at-scale)
  * [Q2: What does it mean that CLIP puts images and text in the "same space"?](#q2-what-does-it-mean-that-clip-puts-images-and-text-in-the-same-space)
  * [Q3: Why must you normalize CLIP embeddings before computing similarity?](#q3-why-must-you-normalize-clip-embeddings-before-computing-similarity)
  * [Q4: Whisper has a 30-second limit. How do you transcribe a 2-hour meeting?](#q4-whisper-has-a-30-second-limit-how-do-you-transcribe-a-2-hour-meeting)
  * [Q5: When should you use multi-modal RAG vs text-only RAG?](#q5-when-should-you-use-multi-modal-rag-vs-text-only-rag)
* [🔨 Practical Practice](#practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 When to Use What](#when-to-use-what)


In [ ]:
# Cell 1 — Install dependencies
# transformers: CLIP and Whisper model loading
# Pillow: image I/O
# sentence-transformers: alternative CLIP wrapper
!pip install -q transformers torch pillow sentence-transformers pydantic numpy

## 1. CLIP: Image-Text Embeddings

### 1.1 How CLIP Works (Contrastive Learning)

CLIP (Contrastive Language-Image Pre-training) was trained on **400 million image-text pairs** scraped from the internet. The training objective is simple but powerful:

```
For each (image, caption) pair in the batch:
1. Encode the image → 768-dim vector
2. Encode the caption → 768-dim vector
3. PUSH matching pairs TOGETHER in vector space
4. PUSH non-matching pairs APART
```

After training, CLIP can:
- Compare ANY image with ANY text by computing cosine similarity
- Classify images with text labels it has **never been trained on** (zero-shot)
- Power cross-modal search in vector databases

| CLIP Variant | Image Encoder | Embedding Dim | Speed | Quality |
|-------------|-------------|:----:|:-----:|:------:|
| `openai/clip-vit-base-patch32` | ViT-B/32 | 512 | ⚡ Fast | Good |
| `openai/clip-vit-base-patch16` | ViT-B/16 | 512 | Medium | Better |
| `openai/clip-vit-large-patch14` | ViT-L/14 | 768 | Slower | **Best** |

### 1.2 Hands-On: Loading CLIP and Computing Embeddings

In [ ]:
# Cell 2 — CLIP: Loading and computing real embeddings
#
# This cell uses the ACTUAL CLIP model from OpenAI via HuggingFace.
# It downloads ~600MB on first run (ViT-B/32 variant for speed).

import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np

# Load CLIP (ViT-B/32 — fast variant for this demo)
model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(model_name)
clip_processor = CLIPProcessor.from_pretrained(model_name)
clip_model.eval()  # Inference mode

print(f"✅ CLIP loaded: {model_name}")
print(f"   Image encoder: {clip_model.config.vision_config.hidden_size}-dim")
print(f"   Text encoder:  {clip_model.config.text_config.hidden_size}-dim")
print(f"   Projection dim: {clip_model.config.projection_dim}")
print(f"   Total params: {sum(p.numel() for p in clip_model.parameters()):,}")

# --- Encode text descriptions ---
descriptions = [
    "a photo of a cat sitting on a windowsill",
    "a diagram of a neural network architecture",
    "a stock market chart showing upward trend",
    "a person coding on a laptop",
    "a sunset over the ocean",
]

with torch.no_grad():
    text_inputs = clip_processor(text=descriptions, return_tensors="pt", padding=True)
    text_embeds = clip_model.get_text_features(**text_inputs)
    # Normalize — CRITICAL for cosine similarity
    text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

print(f"\n📐 Text embeddings shape: {text_embeds.shape}")
print(f"   Each text → a {text_embeds.shape[1]}-dimensional vector")

# --- Compute text-text similarity ---
similarity_matrix = (text_embeds @ text_embeds.T).numpy()
print(f"\n📊 Text-Text Similarity Matrix:")
print(f"{'':32s}", end="")
for i in range(len(descriptions)):
    print(f"  [{i}]", end="")
print()
for i, desc in enumerate(descriptions):
    print(f"  [{i}] {desc[:28]:28s}", end="")
    for j in range(len(descriptions)):
        print(f" {similarity_matrix[i][j]:+.2f}", end="")
    print()

In [ ]:
# Cell 3 — CLIP: Encode synthetic images (without external downloads)
#
# We create simple synthetic images to demonstrate REAL CLIP inference
# without needing to download external images.

from PIL import Image, ImageDraw, ImageFont
import torch

def create_test_image(text_label, bg_color, size=(224, 224)):
    """Create a simple labeled test image."""
    img = Image.new("RGB", size, bg_color)
    draw = ImageDraw.Draw(img)
    # Draw text centered
    bbox = draw.textbbox((0, 0), text_label)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    draw.text(((size[0]-w)//2, (size[1]-h)//2), text_label, fill="white")
    return img

# Create simple test images
test_images = {
    "red square":   create_test_image("RED", (200, 50, 50)),
    "blue circle":  create_test_image("BLUE", (50, 50, 200)),
    "green nature":  create_test_image("GREEN", (50, 180, 50)),
    "dark night":   create_test_image("DARK", (20, 20, 40)),
}

# Encode images with CLIP
candidate_texts = ["a warm red color", "a cool blue color", "nature and greenery", "a dark night scene"]

with torch.no_grad():
    # Encode all images
    image_list = list(test_images.values())
    image_inputs = clip_processor(images=image_list, return_tensors="pt")
    image_embeds = clip_model.get_image_features(**image_inputs)
    image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
    
    # Encode candidate texts
    text_inputs = clip_processor(text=candidate_texts, return_tensors="pt", padding=True)
    text_embeds = clip_model.get_text_features(**text_inputs)
    text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
    
    # Cross-modal similarity
    similarity = (image_embeds @ text_embeds.T).numpy()

print("🔗 Cross-Modal Similarity (Image ↔ Text)")
print("=" * 70)
image_names = list(test_images.keys())
print(f"{'Image':<18}", end="")
for t in candidate_texts:
    print(f"  {t[:14]:14s}", end="")
print()
print("-" * 70)
for i, name in enumerate(image_names):
    print(f"{name:<18}", end="")
    best_j = similarity[i].argmax()
    for j in range(len(candidate_texts)):
        marker = " ⭐" if j == best_j else "   "
        print(f"  {similarity[i][j]:+.3f}{marker:3s}", end="")
    print()

print(f"\n📌 CLIP correctly matches images to their semantic descriptions")
print(f"   even though these are synthetic images it has never seen.")

### 1.3 Zero-Shot Image Classification

CLIP can classify images into **any categories you define at runtime** — no training required. This is zero-shot classification.

In [ ]:
# Cell 4 — Zero-shot classification with CLIP
#
# This is the production pattern for classifying images
# WITHOUT any task-specific training.

def clip_zero_shot_classify(image, candidate_labels, model, processor):
    """Zero-shot image classification using CLIP.
    
    Args:
        image: PIL Image
        candidate_labels: list of text descriptions
        model: CLIPModel
        processor: CLIPProcessor
    Returns:
        dict of {label: probability}
    """
    # Prompt engineering: "a photo of {label}" works better than just "{label}"
    prompts = [f"a photo of {label}" for label in candidate_labels]
    
    with torch.no_grad():
        inputs = processor(
            text=prompts,
            images=image,
            return_tensors="pt",
            padding=True,
        )
        outputs = model(**inputs)
        
        # logits_per_image: (1, num_labels) — similarity scores
        logits = outputs.logits_per_image
        probs = logits.softmax(dim=-1).squeeze().numpy()
    
    return {label: float(prob) for label, prob in zip(candidate_labels, probs)}

# Demo: Classify a synthetic image
test_img = create_test_image("WARM SUNSET", (220, 120, 50))
labels = ["sunset", "ocean", "forest", "city skyline", "snowstorm"]
results = clip_zero_shot_classify(test_img, labels, clip_model, clip_processor)

print("🏷️ Zero-Shot Classification Results")
print("=" * 50)
for label, prob in sorted(results.items(), key=lambda x: -x[1]):
    bar = "█" * int(prob * 40)
    print(f"  {label:15s} {prob:.1%} {bar}")

print(f"\n🎯 Production use cases for zero-shot CLIP:")
print(f"   • Content moderation (NSFW detection)")
print(f"   • Product categorization in e-commerce")
print(f"   • Document type classification (invoice vs receipt vs letter)")
print(f"   • Quality inspection (defective vs OK)")
print(f"\n💡 Cost comparison:")
print(f"   GPT-4V classification: ~$0.01-0.15/image (API call)")
print(f"   CLIP classification:   ~$0.00/image (local, ~50ms on GPU)")

### 1.4 Cross-Modal Search (Image ↔ Text)

The real power of CLIP is enabling **search across modalities**: find images using text queries, or find text using image queries. This is the foundation of multi-modal RAG (Section 4).

In [ ]:
# Cell 5 — Cross-modal search engine with CLIP

class CLIPSearchEngine:
    """Production-style cross-modal search using CLIP embeddings.
    
    In production, you'd store embeddings in a vector DB (ChromaDB, Pinecone)
    as shown in NB21. Here we use in-memory for demonstration.
    """
    
    def __init__(self, model, processor):
        self.model = model
        self.processor = processor
        self.index = []     # list of {type, content, embedding, metadata}
    
    def add_image(self, image, metadata=None):
        """Index an image."""
        with torch.no_grad():
            inputs = self.processor(images=image, return_tensors="pt")
            embed = self.model.get_image_features(**inputs)
            embed = embed / embed.norm(dim=-1, keepdim=True)
        self.index.append({"type": "image", "content": image, "embedding": embed.squeeze(), "metadata": metadata or {}})
    
    def add_text(self, text, metadata=None):
        """Index a text document."""
        with torch.no_grad():
            inputs = self.processor(text=[text], return_tensors="pt", padding=True)
            embed = self.model.get_text_features(**inputs)
            embed = embed / embed.norm(dim=-1, keepdim=True)
        self.index.append({"type": "text", "content": text, "embedding": embed.squeeze(), "metadata": metadata or {}})
    
    def search_by_text(self, query, top_k=5):
        """Search all indexed items (images AND text) using a text query."""
        with torch.no_grad():
            inputs = self.processor(text=[query], return_tensors="pt", padding=True)
            q_embed = self.model.get_text_features(**inputs)
            q_embed = q_embed / q_embed.norm(dim=-1, keepdim=True)
        
        results = []
        for item in self.index:
            sim = torch.dot(q_embed.squeeze(), item["embedding"]).item()
            results.append({**item, "similarity": sim})
        
        results.sort(key=lambda x: x["similarity"], reverse=True)
        return results[:top_k]

# Build a multi-modal search index
engine = CLIPSearchEngine(clip_model, clip_processor)

# Index images
engine.add_image(create_test_image("CHART UP", (50, 150, 50)), {"name": "revenue_chart", "type": "chart"})
engine.add_image(create_test_image("ARCHITECTURE", (50, 50, 150)), {"name": "system_diagram", "type": "diagram"})
engine.add_image(create_test_image("TEAM PHOTO", (150, 100, 50)), {"name": "team_photo", "type": "photo"})

# Index text documents
engine.add_text("Q4 revenue grew 15% to $58.1M driven by enterprise sales.", {"name": "quarterly_report"})
engine.add_text("The microservices architecture uses Kubernetes for orchestration.", {"name": "tech_doc"})
engine.add_text("The engineering team expanded to 67 members across 4 locations.", {"name": "hr_update"})

# Search!
queries = ["financial performance and revenue growth", "system architecture diagram", "team and people"]
print("🔍 Multi-Modal Search Results")
print("=" * 60)

for query in queries:
    results = engine.search_by_text(query, top_k=3)
    print(f"\n  Query: \"{query}\"")
    for r in results:
        icon = "🖼️" if r["type"] == "image" else "📝"
        name = r["metadata"].get("name", "?")
        content = name if r["type"] == "image" else r["content"][:50]
        print(f"    {icon} sim={r['similarity']:+.3f} | {content}")

print(f"\n📌 CLIP searches BOTH images and text in a single query.")
print(f"   This powers multi-modal RAG (Section 4).")

---

## 2. Whisper: Production Speech-to-Text

### 2.1 Whisper Architecture Overview

Whisper is an encoder-decoder Transformer (like the Seq2Seq models in NB16_04) trained on **680,000 hours** of multilingual audio.

```
Audio waveform → Mel spectrogram → Encoder (Transformer) → Decoder (Transformer) → Text
```

| Whisper Model | Parameters | VRAM | Speed (RTF) | Quality |
|:-------------|:---------:|:----:|:----------:|:-------:|
| `tiny` | 39M | ~1 GB | 32× | Basic |
| `base` | 74M | ~1 GB | 16× | Good |
| `small` | 244M | ~2 GB | 6× | Better |
| `medium` | 769M | ~5 GB | 2× | Great |
| `large-v3` | 1.5B | ~10 GB | 1× | **Best** |

*RTF = Real-Time Factor (32× means 32 seconds of audio processed per second)*

### 2.2 Hands-On: Transcription with Whisper

In [ ]:
# Cell 6 — Whisper: Real model loading and transcription pipeline
#
# We load the ACTUAL Whisper model and show the real API.
# Since we can't record audio in a notebook, we synthesize a test waveform
# and demonstrate the full pipeline.

from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import numpy as np

# Load Whisper tiny (39M params — fast for demo)
whisper_model_name = "openai/whisper-tiny"
whisper_processor = WhisperProcessor.from_pretrained(whisper_model_name)
whisper_model = WhisperForConditionalGeneration.from_pretrained(whisper_model_name)
whisper_model.eval()

print(f"✅ Whisper loaded: {whisper_model_name}")
print(f"   Parameters: {sum(p.numel() for p in whisper_model.parameters()):,}")
print(f"   Sample rate: {whisper_processor.feature_extractor.sampling_rate} Hz")
print(f"   Max audio length: 30 seconds per chunk")

# --- Demonstrate the transcription pipeline ---
# Generate a synthetic audio waveform (sine wave — Whisper will attempt to decode it)
sample_rate = 16000  # Whisper expects 16kHz
duration = 3  # seconds
t = np.linspace(0, duration, sample_rate * duration)
# Mix of frequencies to simulate speech-like audio
audio = 0.5 * np.sin(2 * np.pi * 440 * t) + 0.3 * np.sin(2 * np.pi * 880 * t)
audio = audio.astype(np.float32)

# Process through Whisper pipeline
input_features = whisper_processor(
    audio, 
    sampling_rate=sample_rate, 
    return_tensors="pt"
).input_features

print(f"\n📊 Audio Processing:")
print(f"   Raw audio shape:       ({len(audio)},) = {duration}s at {sample_rate}Hz")
print(f"   Mel spectrogram shape: {input_features.shape}")
print(f"   (batch, mel_bins, time_frames)")

# Generate transcription
with torch.no_grad():
    predicted_ids = whisper_model.generate(
        input_features,
        language="en",
        task="transcribe",
    )
    transcription = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)

print(f"\n🎤 Transcription result: '{transcription[0]}'")
print(f"   (Synthetic sine wave — real speech would produce real words)")

print(f"\n📌 Production usage with real audio files:")
production_code = '''
import librosa  # or soundfile, torchaudio

# Load audio file
audio, sr = librosa.load("meeting.mp3", sr=16000)

# Transcribe
inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
    ids = model.generate(inputs.input_features, language="en")
text = processor.batch_decode(ids, skip_special_tokens=True)[0]
'''
print(production_code)

### 2.3 Production Audio Pipeline

Whisper has a **30-second limit** per chunk. For longer audio (meetings, podcasts), you need a chunking strategy:

In [ ]:
# Cell 7 — Production audio pipeline with chunking

class WhisperProductionPipeline:
    """Production-grade audio transcription with chunking.
    
    Handles arbitrarily long audio by splitting into 30-second chunks
    with overlap to prevent word splitting at boundaries.
    """
    
    def __init__(self, model, processor, chunk_length_s=30, stride_s=5):
        self.model = model
        self.processor = processor
        self.sample_rate = 16000
        self.chunk_length = chunk_length_s * self.sample_rate
        self.stride = stride_s * self.sample_rate
    
    def transcribe(self, audio_array, language="en"):
        """Transcribe long audio by chunking."""
        total_samples = len(audio_array)
        duration_s = total_samples / self.sample_rate
        
        # Split into overlapping chunks
        chunks = []
        start = 0
        while start < total_samples:
            end = min(start + self.chunk_length, total_samples)
            chunks.append(audio_array[start:end])
            start += self.chunk_length - self.stride  # Overlap
        
        # Transcribe each chunk
        transcripts = []
        for i, chunk in enumerate(chunks):
            inputs = self.processor(chunk, sampling_rate=self.sample_rate, return_tensors="pt")
            with torch.no_grad():
                ids = self.model.generate(inputs.input_features, language=language)
                text = self.processor.batch_decode(ids, skip_special_tokens=True)[0]
                transcripts.append(text.strip())
        
        return {
            "text": " ".join(transcripts),
            "duration_s": duration_s,
            "num_chunks": len(chunks),
            "chunk_length_s": self.chunk_length // self.sample_rate,
        }

# Demo with 2-minute synthetic audio
pipeline = WhisperProductionPipeline(whisper_model, whisper_processor)

long_audio = np.random.randn(16000 * 120).astype(np.float32) * 0.01  # 2 min silence/noise
result = pipeline.transcribe(long_audio)

print("🎙️ Production Audio Pipeline")
print("=" * 50)
print(f"  Audio duration:  {result['duration_s']:.0f}s ({result['duration_s']/60:.1f} min)")
print(f"  Chunks created:  {result['num_chunks']} × {result['chunk_length_s']}s")
print(f"  Transcription:   '{result['text'][:100]}...'")

# Cost analysis
print(f"\n💰 Cost Analysis (Whisper API pricing):")
for duration_min in [5, 30, 60, 480]:
    api_cost = duration_min * 0.006
    gpu_cost = duration_min / 60 * 0.50  # ~$0.50/hr on A10G
    print(f"  {duration_min:4d} min: API=${api_cost:.2f}  |  Self-hosted=${gpu_cost:.2f}")
print(f"\n📌 Self-hosting breaks even at ~100 min/day.")

---

## 3. Vision API Patterns (GPT-4V / Gemini)

### 3.1 Image Input Formats and Cost Optimization

When using commercial Vision APIs, image format dramatically affects cost:

| Method | Token Cost | When to Use |
|--------|:---------:|-------------|
| URL (low detail) | ~85 tokens | Simple classification, yes/no questions |
| URL (high detail) | ~765 tokens/tile | OCR, detailed extraction |
| Base64 (low detail) | ~85 tokens | Local files, simple tasks |
| Base64 (high detail) | ~765 tokens/tile | Local files, complex analysis |

### 3.2 Structured Extraction from Images

In [ ]:
# Cell 8 — Vision API patterns
#
# This shows the EXACT request format for GPT-4V / Gemini Vision APIs.
# No API key needed — we build the request objects.

import base64, json
from pydantic import BaseModel, Field
from typing import List
from io import BytesIO

def image_to_base64(image: Image.Image) -> str:
    """Convert PIL Image → base64 string for API calls."""
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

# Structured output schema (from NB19)
class InvoiceExtraction(BaseModel):
    vendor_name: str = Field(description="Name of the vendor/company")
    invoice_number: str = Field(description="Invoice reference number")
    date: str = Field(description="Invoice date in YYYY-MM-DD format")
    total_amount: float = Field(description="Total amount due")
    currency: str = Field(default="USD")
    line_items: List[dict] = Field(description="List of items with description, qty, price")

def build_vision_request(image, prompt, detail="auto", schema=None):
    """Build a multimodal API request (OpenAI format)."""
    b64 = image_to_base64(image) if isinstance(image, Image.Image) else image
    
    request = {
        "model": "gpt-4o",
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {
                    "url": f"data:image/png;base64,{b64[:20]}...",
                    "detail": detail
                }},
            ]
        }],
        "max_tokens": 1000,
    }
    if schema:
        request["response_format"] = {"type": "json_schema", "json_schema": schema}
    return request

# Production use cases
test_img = create_test_image("INVOICE", (240, 240, 240))

use_cases = [
    {
        "name": "Invoice Processing",
        "prompt": "Extract all fields from this invoice. Return JSON.",
        "detail": "high",
        "schema": InvoiceExtraction.model_json_schema(),
    },
    {
        "name": "Quality Inspection",
        "prompt": "Does this product photo show any defects? Answer: yes/no, defect_type, severity.",
        "detail": "high",
        "schema": None,
    },
    {
        "name": "Dashboard Analysis",
        "prompt": "What metrics are shown? Are any in warning/critical state?",
        "detail": "low",
        "schema": None,
    },
]

print("🖼️ Vision API Use Cases")
print("=" * 60)
for uc in use_cases:
    req = build_vision_request(test_img, uc["prompt"], uc["detail"], uc.get("schema"))
    est_tokens = 85 if uc["detail"] == "low" else 765
    est_cost = est_tokens * 0.01 / 1000  # $0.01/1K tokens
    has_schema = "✅" if uc.get("schema") else "❌"
    print(f"\n  📋 {uc['name']}")
    print(f"     Detail: {uc['detail']} (~{est_tokens} tokens, ~${est_cost:.4f})")
    print(f"     Structured output: {has_schema}")
    print(f"     Prompt: \"{uc['prompt'][:60]}\"")

print(f"\n💡 Cost optimization rules:")
print(f"   1. Use 'low' detail for classification (10× cheaper)")
print(f"   2. Use structured output (NB19) for guaranteed JSON")
print(f"   3. Batch images when possible (reduces API overhead)")
print(f"   4. Use CLIP for simple tasks, Vision API for complex reasoning")

---

## 4. Multi-Modal RAG

Traditional RAG (NB22) searches text chunks with text queries. **Multi-modal RAG** extends this:

```
Document (PDF with text + images + charts)
    │
    ├── Text chunks  → Text embeddings (NB22 approach)
    └── Image chunks → CLIP embeddings (THIS notebook)
           │
           └── ALL stored in the same vector DB
                   │
                   └── Query retrieves BOTH text AND images
                           │
                           └── LLM sees text + images in context
```

In [ ]:
# Cell 9 — Multi-modal RAG pipeline

class MultiModalRAG:
    """RAG pipeline that retrieves both text and images.
    
    Extends NB22's text-only RAG with CLIP image embeddings.
    In production, use ChromaDB (NB21) as the vector store.
    """
    
    def __init__(self, clip_model, clip_processor):
        self.search = CLIPSearchEngine(clip_model, clip_processor)
        self.documents = []
    
    def ingest_document(self, text_chunks, images_with_captions):
        """Ingest a document's text and images into the index."""
        for chunk in text_chunks:
            self.search.add_text(chunk, {"source": "text"})
        for img, caption in images_with_captions:
            self.search.add_image(img, {"source": "image", "caption": caption})
        self.documents.append({"text_chunks": len(text_chunks), "images": len(images_with_captions)})
    
    def query(self, question, top_k=3):
        """Retrieve relevant text and images for a question."""
        results = self.search.search_by_text(question, top_k=top_k)
        
        # Separate text and image results
        context = {"text_chunks": [], "images": [], "query": question}
        for r in results:
            if r["type"] == "text":
                context["text_chunks"].append({"text": r["content"], "sim": r["similarity"]})
            else:
                context["images"].append({"caption": r["metadata"].get("caption", "?"), "sim": r["similarity"]})
        return context
    
    def build_llm_prompt(self, context):
        """Build a prompt that includes both text and image context."""
        prompt = f"Question: {context['query']}\n\nContext:\n"
        for tc in context["text_chunks"]:
            prompt += f"[TEXT] {tc['text']}\n"
        for img in context["images"]:
            prompt += f"[IMAGE] {img['caption']} (would be sent as base64 to vision API)\n"
        prompt += "\nAnswer using the context above:"
        return prompt

# Build a multi-modal RAG system
rag = MultiModalRAG(clip_model, clip_processor)

# Ingest a simulated quarterly report
rag.ingest_document(
    text_chunks=[
        "Q4 2025 revenue reached $58.1M, a 15% increase year-over-year.",
        "Customer acquisition cost decreased to $142 per customer.",
        "Engineering headcount grew to 67 across 4 global offices.",
        "The company launched 3 new product features in Q4.",
    ],
    images_with_captions=[
        (create_test_image("REVENUE UP", (50, 180, 50)), "Revenue growth chart Q1-Q4 2025"),
        (create_test_image("ORG CHART", (50, 50, 180)), "Engineering team organizational structure"),
        (create_test_image("PRODUCT", (180, 50, 50)), "New product feature screenshots"),
    ]
)

# Query the multi-modal RAG
questions = [
    "What was the revenue performance in Q4?",
    "Show me the team structure",
    "What new features were launched?",
]

print("🔗 Multi-Modal RAG Pipeline")
print("=" * 60)
for q in questions:
    context = rag.query(q, top_k=3)
    prompt = rag.build_llm_prompt(context)
    print(f"\n  ❓ {q}")
    print(f"     Retrieved: {len(context['text_chunks'])} text + {len(context['images'])} images")
    for tc in context["text_chunks"]:
        print(f"     📝 sim={tc['sim']:+.3f} | {tc['text'][:60]}")
    for img in context["images"]:
        print(f"     🖼️ sim={img['sim']:+.3f} | {img['caption']}")

print(f"\n📌 In production, images are sent as base64 to GPT-4V alongside text.")
print(f"   The LLM can reason about charts, diagrams, AND text simultaneously.")

---

## ✅ Knowledge Check

### Q1: Why is CLIP better than GPT-4V for image classification at scale?
<details><summary>👉 Answer</summary>

CLIP runs locally, costs $0/image, and processes images in ~50ms on GPU. GPT-4V costs $0.01-0.15/image and takes 1-3 seconds per request. For 100K images, CLIP costs $0 and takes ~80 minutes. GPT-4V costs $1,000-15,000 and takes 28-83 hours. CLIP wins massively for batch classification. Use GPT-4V only when you need *reasoning* about image content.
</details>

### Q2: What does it mean that CLIP puts images and text in the "same space"?
<details><summary>👉 Answer</summary>

CLIP's image encoder and text encoder both output vectors of the same dimensionality (e.g., 512 or 768). During training, matching image-text pairs are pushed to have high cosine similarity, while non-matching pairs are pushed apart. After training, you can directly compare an image vector with a text vector using cosine similarity — they share the same semantic space. This enables cross-modal search: find images with text queries or vice versa.
</details>

### Q3: Why must you normalize CLIP embeddings before computing similarity?
<details><summary>👉 Answer</summary>

CLIP was trained using cosine similarity (the dot product of unit vectors). Raw embeddings have varying magnitudes. Without normalization, the dot product mixes magnitude with direction — a long vector pointing slightly wrong could score higher than a short vector pointing exactly right. L2 normalization ensures all vectors have magnitude 1, so the dot product equals cosine similarity.
</details>

### Q4: Whisper has a 30-second limit. How do you transcribe a 2-hour meeting?
<details><summary>👉 Answer</summary>

Split the audio into **overlapping 30-second chunks** (e.g., 30s chunks with 5s overlap). Transcribe each chunk independently, then concatenate the results. The overlap prevents words from being cut at chunk boundaries. In the HuggingFace pipeline, use `chunk_length_s=30` and `stride_length_s=(5, 5)` for automatic chunking. For speaker diarization (who said what), combine Whisper with a tool like `pyannote.audio`.
</details>

### Q5: When should you use multi-modal RAG vs text-only RAG?
<details><summary>👉 Answer</summary>

Use **multi-modal RAG** when your documents contain visual information that text alone can't capture: charts (the shape of a trend matters), diagrams (architecture flows), screenshots (UI state), photos (product quality). Use **text-only RAG** when your corpus is purely textual (code repos, API docs, legal contracts). The key question: "Would a human need to SEE the document to answer this question?"
</details>

---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Load `openai/clip-vit-large-patch14` (the high-quality variant) and compare its embeddings with the `vit-base-patch32` used above. Which produces more distinct clusters for similar concepts?
2. Use Whisper to transcribe a real audio file from your computer. Measure the processing time and calculate the real-time factor.

### Tier 2: Intermediate
1. **Build an Image Search Engine:** Download 50 images from Unsplash. Index them with CLIP. Build a text-search interface that returns the top-5 matching images for any query. Measure retrieval quality.
2. **Meeting Assistant Pipeline:** Combine Whisper transcription with an LLM via LiteLLM (NB20) to build a pipeline that: transcribes audio → extracts action items → generates a summary email.

### Tier 3: Advanced
1. **Multi-Modal RAG with ChromaDB:** Extend the Section 4 RAG pipeline to use ChromaDB (NB21) as the vector store. Ingest a real PDF with images (using `pymupdf` or `pdf2image`). Query with text and retrieve both text chunks and relevant images.
2. **Fine-Tune CLIP:** Use the `sentence-transformers` library to fine-tune CLIP on a domain-specific dataset (e.g., medical images + captions). Compare zero-shot vs fine-tuned performance on a classification task.

---

## 🎯 Summary & Key Takeaways

| Concept | Model | Key Takeaway |
|---------|-------|-------------|
| Image-text embeddings | CLIP | Same vector space → cross-modal search, zero-shot classification |
| Speech-to-text | Whisper | 680K hours of training, open source, chunk for long audio |
| Vision APIs | GPT-4V/Gemini | Complex reasoning, use `detail=low` for cost savings |
| Multi-modal RAG | CLIP + ChromaDB + LLM | Index images alongside text, retrieve both |

### 🧠 When to Use What

```
Simple image classification?     → CLIP (free, local, 50ms)
Complex image reasoning?         → GPT-4V / Gemini (API, $0.01-0.15)
Image search in vector DB?       → CLIP embeddings + ChromaDB
Speech-to-text?                  → Whisper (tiny for speed, large-v3 for quality)
Document with charts/images?     → Multi-modal RAG (CLIP + text embeddings)
```

**Next →** `24_agentic_orchestration.ipynb` — Stateful Agentic Orchestration & Memory Architectures